# Avaliacao de Comites de Classificacao
## Imports e funções

In [3]:
import traceback
import pandas as pd
import numpy  as np
from sklearn.metrics          import confusion_matrix, f1_score
from scipy.stats              import friedmanchisquare
from scikit_posthocs          import posthoc_nemenyi_friedman
from openpyxl                 import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

versao_execucao = "v20251102-1210"
precisao = 18

def preparar_dados_para_testes_estatisticos(df):
    """
    Prepara os dados para testes de Friedman e Nemenyi comparando configurações dentro de cada modelo.
    
    Args:
        df: DataFrame com os resultados dos modelos
        
    Returns:
        dict: Dicionário onde cada chave é o model_name (KNN, Decision Tree, etc.) 
              e o valor é uma matriz numpy preparada para os testes estatísticos.
              A matriz tem:
              - Linhas: combinações de dataset × training_type  
              - Colunas: configurações disponíveis para aquele modelo (dinâmico)
              - Valores: f1_scores
    """
    
    # Dicionário para armazenar os resultados por modelo
    dados_preparados = {}
    
    # Obter lista de modelos únicos
    modelos_unicos = df['model_name'].unique()
    
    # Para cada modelo
    for model_name in modelos_unicos:
        # Filtrar dados apenas deste modelo
        df_modelo = df[df['model_name'] == model_name].copy()
        
        # Obter datasets e training_types únicos, ordenados
        datasets_ordenados = sorted(df_modelo['dataset'].unique())
        training_types_ordenados = sorted(df_modelo['training_type'].unique())
        
        # Obter configurações disponíveis para este modelo (ordenadas)
        configs_disponiveis = sorted(df_modelo['config_rank'].unique())
        
        # Lista para armazenar as linhas da matriz
        matriz_dados = []
        
        # Para cada combinação dataset × training_type
        for dataset in datasets_ordenados:
            for training_type in training_types_ordenados:
                # Filtrar dados desta combinação
                mask = (df_modelo['dataset'] == dataset) & (df_modelo['training_type'] == training_type)
                df_comb = df_modelo[mask]
                
                # Coletar f1_scores das configurações disponíveis
                f1_scores = []
                for config_rank in configs_disponiveis:
                    score = df_comb[df_comb['config_rank'] == config_rank]['f1_score']
                    if len(score) > 0:
                        f1_scores.append(score.iloc[0])
                    else:
                        f1_scores.append(np.nan)  # Valor ausente
                
                # Adicionar esta linha à matriz
                matriz_dados.append(f1_scores)
        
        # Converter para array numpy
        dados_preparados[model_name] = np.array(matriz_dados)
    
    return dados_preparados


def executar_testes_estatisticos(dados_preparados):
    """
    Executa testes de Friedman e Nemenyi para todos os modelos nos dados preparados.
    
    Args:
        dados_preparados: Dicionário retornado por preparar_dados_para_testes_estatisticos()
        
    Returns:
        dict: Dicionário contendo os resultados dos testes para cada modelo
    """
    
    resultados = {}
    
    for model_name, matriz_dados in dados_preparados.items():
        print(f"\n{'='*50}")
        print(f"PROCESSANDO MODELO: {model_name}")
        print(f"{'='*50}")
        
        # Verificar se há dados suficientes
        if matriz_dados.shape[0] < 3:
            print(f"ERRO: Dados insuficientes para {model_name} ({matriz_dados.shape[0]} cenários)")
            print("Friedman e Nemenyi precisam de pelo menos 3 observações")
            continue
            
        if matriz_dados.shape[1] < 2:
            print(f"ERRO: {model_name} tem apenas {matriz_dados.shape[1]} configuração")
            print("Precisa de pelo menos 2 configurações para comparar")
            continue
        
        # Remover linhas com NaN
        matriz_limpa = matriz_dados[~np.isnan(matriz_dados).any(axis=1)]
        
        if matriz_limpa.shape[0] < 3:
            print(f"ERRO: Após remover NaN, restaram {matriz_limpa.shape[0]} cenários válidos")
            print("Friedman e Nemenyi precisam de pelo menos 3 observações")
            continue
        
        print(f"Dados válidos: {matriz_limpa.shape[0]} cenários × {matriz_limpa.shape[1]} configurações")
        
        # Aplicar teste de Friedman
        try:
            friedman_stat, friedman_p = friedmanchisquare(*matriz_limpa.T)
            print(f"\nFriedman Test:")
            print(f"  Estatística: {friedman_stat:.{precisao}f}")  # Usa a variável precisao = 12
            print(f"  p-valor: {friedman_p:.{precisao}f}")
            
            if friedman_p < 0.05:
                print("  → DIFERENÇAS SIGNIFICATIVAS encontradas entre configurações")
            else:
                print("  → NÃO há diferenças significativas entre configurações")
                
        except Exception as e:
            print(f"ERRO no teste de Friedman: {e}")
            friedman_stat, friedman_p = None, None
        
        # Aplicar teste de Nemenyi (sempre, independente do p-valor)
        try:
            print("Aplicando teste de Nemenyi...")
            nemenyi_result = posthoc_nemenyi_friedman(matriz_limpa)
            
            # Renomear índices e colunas para ficar mais claro
            num_configs = matriz_limpa.shape[1]
            nomes_configs = [f"config_{i+1}" for i in range(num_configs)]
            nemenyi_result.index = nomes_configs
            nemenyi_result.columns = nomes_configs
            
            print("Resultado Nemenyi (p-valores ajustados):")
            print(nemenyi_result.round(precisao))
            # print(nemenyi_result)
            
            # Identificar pares com diferenças significativas (p < 0.05)
            pares_significativos = []
            for i in range(len(nemenyi_result)):
                for j in range(i+1, len(nemenyi_result)):
                    if nemenyi_result.iloc[i, j] < 0.05:
                        pares_significativos.append((nomes_configs[i], nomes_configs[j]))
            
            if pares_significativos:
                print(f"\nPares com diferenças significativas (p < 0.05):")
                for par in pares_significativos:
                    p_valor = nemenyi_result.loc[par[0], par[1]]
                    print("  {} vs {}: p = {:.{}f}".format(par[0], par[1], p_valor, precisao))
                    # print(f"  {par[0]} vs {par[1]}: p = {p_valor:.{precisao}f}")
            else:
                print("\nNenhum par apresenta diferenças significativas")
                
        except Exception as e:
            print(f"ERRO no teste de Nemenyi: {e}")
            nemenyi_result = None
        
        # Armazenar resultados
        resultados[model_name] = {
            'friedman_stat': friedman_stat,
            'friedman_p_value': friedman_p,
            'nemenyi_matrix': nemenyi_result,
            'num_cenarios': matriz_limpa.shape[0],
            'num_configuracoes': matriz_limpa.shape[1],
            'pares_significativos': pares_significativos if 'pares_significativos' in locals() else []
        }
        
        print(f"\n{'='*50}")
    
    return resultados

def salvar_resultados_excel(resultados_testes, dados_preparados, nome_arquivo):
    """
    Salva os resultados dos testes estatísticos em um arquivo Excel para análise.
    
    Args:
        resultados_testes: Dicionário retornado por executar_testes_estatisticos()
        dados_preparados: Dicionário retornado por preparar_dados_para_testes_estatisticos()
        nome_arquivo: Nome do arquivo Excel a ser criado
    """
    
    # Criar workbook
    wb = Workbook()
    
    # ===== ABA 1: RESUMO GERAL =====
    ws_resumo = wb.active
    ws_resumo.title = "Resumo_Geral"
    
    # Preparar dados do resumo
    dados_resumo = []
    for model_name, resultados in resultados_testes.items():
        # Calcular médias de desempenho
        matriz_modelo = dados_preparados[model_name]
        matriz_limpa = matriz_modelo[~np.isnan(matriz_modelo).any(axis=1)]
        
        medias_configuracoes = {}
        for i in range(matriz_limpa.shape[1]):
            config_name = f"config_{i+1}"
            media_f1 = np.mean(matriz_limpa[:, i])
            medias_configuracoes[config_name] = media_f1
        
        # Ranking das configurações
        ranking = sorted(medias_configuracoes.items(), key=lambda x: x[1], reverse=True)
        
        # Verificar diferenças significativas
        diferencas_significativas = (resultados['friedman_p_value'] is not None and 
                                    resultados['friedman_p_value'] < 0.05 and 
                                    resultados['pares_significativos'])
        
        dados_resumo.append({
            'Modelo': model_name,
            'Cenários_Analisados': resultados['num_cenarios'],
            'Configurações_Comparadas': resultados['num_configuracoes'],
            'Friedman_Estatística': resultados['friedman_stat'],
            'Friedman_P_Valor': resultados['friedman_p_value'],
            'Diferenças_Significativas': 'Sim' if diferencas_significativas else 'Não',
            'Melhor_Configuração': ranking[0][0] if ranking else None,
            'Melhor_F1_Médio': ranking[0][1] if ranking else None,
            'Segunda_Melhor_Config': ranking[1][0] if len(ranking) > 1 else None,
            'Segunda_F1_Médio': ranking[1][1] if len(ranking) > 1 else None,
            'Terceira_Melhor_Config': ranking[2][0] if len(ranking) > 2 else None,
            'Terceira_F1_Médio': ranking[2][1] if len(ranking) > 2 else None,
            'Pares_Significativos': len(resultados['pares_significativos']) if resultados['pares_significativos'] else 0
        })
    
    # Criar DataFrame do resumo
    df_resumo = pd.DataFrame(dados_resumo)
    
    # Adicionar ao Excel
    for r in dataframe_to_rows(df_resumo, index=False, header=True):
        ws_resumo.append(r)
    
    # ===== ABA 2: PARES SIGNIFICATIVOS =====
    ws_pares = wb.create_sheet("Pares_Significativos")
    
    # Preparar dados dos pares significativos
    dados_pares = []
    for model_name, resultados in resultados_testes.items():
        if resultados['pares_significativos']:
            for i, (config_a, config_b) in enumerate(resultados['pares_significativos'], 1):
                p_valor = resultados['nemenyi_matrix'].loc[config_a, config_b]
                dados_pares.append({
                    'Modelo': model_name,
                    'Par_Numero': i,
                    'Configuração_A': config_a,
                    'Configuração_B': config_b,
                    'P_Valor': p_valor,
                    'Diferença_Significativa': 'Sim' if p_valor < 0.05 else 'Não'
                })
        else:
            dados_pares.append({
                'Modelo': model_name,
                'Par_Numero': None,
                'Configuração_A': None,
                'Configuração_B': None,
                'P_Valor': None,
                'Diferença_Significativa': 'Nenhum par significativo'
            })
    
    df_pares = pd.DataFrame(dados_pares)
    for r in dataframe_to_rows(df_pares, index=False, header=True):
        ws_pares.append(r)
    
    # ===== ABA 3: MATRIZES NEMENYI (uma aba por modelo) =====
    for model_name, resultados in resultados_testes.items():
        if resultados['nemenyi_matrix'] is not None:
            ws_nemenyi = wb.create_sheet(f"Nemenyi_{model_name.replace(' ', '_')}")
            
            # Adicionar título
            ws_nemenyi.append([f"Matriz Nemenyi - {model_name}"])
            ws_nemenyi.append([])  # Linha vazia
            
            # Adicionar a matriz
            df_nemenyi = resultados['nemenyi_matrix'].round(precisao)
            # df_nemenyi = resultados['nemenyi_matrix']
            for r in dataframe_to_rows(df_nemenyi, index=True, header=True):
                ws_nemenyi.append(r)
    
    # Salvar o arquivo
    wb.save(nome_arquivo)
    print(f"✅ Resultados salvos em: {nome_arquivo}")
    
    # Mostrar resumo do que foi salvo
    print(f"📊 Arquivo contém:")
    print(f"   • Aba 'Resumo_Geral': {len(df_resumo)} comites resumidos")
    print(f"   • Aba 'Pares_Significativos': {len(df_pares)} pares analisados")
    print(f"   • {len([m for m in resultados_testes.keys() if resultados_testes[m]['nemenyi_matrix'] is not None])} abas de matrizes Nemenyi")


## Carregando resultados

In [6]:
df = pd.read_csv(f'resultados_todos_comites.csv')

df

,dataset,model,model_name,config_rank,params,training_type,f1_score,f1_std,scale_data,execution_time
0,hogfeat_128_16_4_9_pca,Bagging,Bagging,1,"{'estimatorModel': 'knn', 'n_estimators': 10}",holdout,0.748561,0.000000,False,0.409672
1,hogfeat_128_16_4_9_pca,Bagging,Bagging,1,"{'estimatorModel': 'knn', 'n_estimators': 10}",crossvalidation,0.718662,0.030141,False,0.630306
2,hogfeat_128_16_4_9_pca,Bagging,Bagging,2,"{'estimatorModel': 'knn', 'n_estimators': 20}",holdout,0.774212,0.000000,False,0.080774
3,hogfeat_128_16_4_9_pca,Bagging,Bagging,2,"{'estimatorModel': 'knn', 'n_estimators': 20}",crossvalidation,0.726323,0.039484,False,1.262660
4,hogfeat_128_16_4_9_pca,Bagging,Bagging,3,"{'estimatorModel': 'knn', 'n_estimators': 30}",holdout,0.767689,0.000000,False,0.090486
...,...,...,...,...,...,...,...,...,...,...
763,lbpfeat_256_24_192,Stacking,Stacking,2,{'n_estimators': 10},crossvalidation,0.736695,0.064490,False,156.037320
764,lbpfeat_256_24_192,Stacking,Stacking,3,{'n_estimators': 15},holdout,0.756873,0.000000,False,7.795595
765,lbpfeat_256_24_192,Stacking,Stacking,3,{'n_estimators': 15},crossvalidation,0.727398,0.059883,False,182.978482
766,lbpfeat_256_24_192,Stacking,Stacking,4,{'n_estimators': 20},holdout,0.731937,0.000000,False,10.477772


## Preparando os dados para Friedman e Nemenyi

In [7]:

dados_preparados = preparar_dados_para_testes_estatisticos(df)

    # # Verificar o resultado
    # print("Shape do resultado:", resultado.shape)
    # print("Colunas do resultado:", list(resultado.columns))
    # print("Primeiras linhas do resultado:")
    # print(resultado.head())
# Verificar as dimensões para cada modelo
for model_name, matriz in dados_preparados.items():
    print(f"{model_name}: {matriz.shape[0]} cenários × {matriz.shape[1]} configurações")


Bagging: 24 cenários × 12 configurações
Random Forest: 24 cenários × 12 configurações
Voting: 24 cenários × 4 configurações
Stacking: 24 cenários × 4 configurações


## Exibindo os resultado de cada modelo

In [11]:
resultados_testes = executar_testes_estatisticos(dados_preparados)

# Loop para exibir resultados de cada modelo
print("=" * 80)
print("RESUMO DOS TESTES ESTATÍSTICOS POR MODELO")
print("=" * 80)

for model_name, resultados in resultados_testes.items():
    print(f"\n🔍 MODELO: {model_name}")
    print("-" * 50)
    
    # Informações básicas
    print(f"📊 Cenários analisados: {resultados['num_cenarios']}")
    print(f"⚙️  Configurações comparadas: {resultados['num_configuracoes']}")
    
    # Resultados do Friedman
    if resultados['friedman_stat'] is not None:
        print(f"\n🎯 TESTE DE FRIEDMAN:")
        formato = f".{precisao}f"
        print(f"   Estatística: {resultados['friedman_stat']:{formato}}")
        print(f"   p-valor: {resultados['friedman_p_value']:{formato}}")        # print(f"   Estatística: {resultados['friedman_stat']:.{precisao}f}")
        # print(f"   p-valor: {resultados['friedman_p_value']:.{precisao}f}")
        
        if resultados['friedman_p_value'] < 0.05:
            print("   ✅ DIFERENÇAS SIGNIFICATIVAS entre configurações")
        else:
            print("   ❌ NÃO há diferenças significativas entre configurações")
    else:
        print("   ❌ Erro no teste de Friedman")
    
    # Resultados do Nemenyi
    if resultados['nemenyi_matrix'] is not None:
        print(f"\n📋 TESTE DE NEMENYI (p-valores ajustados):")
        print("   Matriz de comparações:")
        # Exibir apenas valores significativos para não poluir a saída
        nemenyi_df = resultados['nemenyi_matrix'].copy()
        # Formatar para melhor visualização
        pd.set_option('display.float_format', f'{{:.{precisao}f}}'.format)
        # pd.set_option('display.float_format', '{:.{precisao}f}'.format)
        print(nemenyi_df.to_string())
        
        # Resumo dos pares significativos
        if resultados['pares_significativos']:
            print(f"\n⚠️  PARES COM DIFERENÇAS SIGNIFICATIVAS (p < 0.05):")
            for i, (config_a, config_b) in enumerate(resultados['pares_significativos'], 1):
                p_valor = resultados['nemenyi_matrix'].loc[config_a, config_b]
                print(f"   {i}. {config_a} vs {config_b}: p = {p_valor:.{precisao}f}")
        else:
            print("   ✅ Nenhum par apresenta diferenças significativas")
    else:
        print("   ❌ Erro no teste de Nemenyi")
    
    # ANÁLISE DE DESEMPENHO (sempre mostrar TOP 3)
    print(f"\n📊 ANÁLISE DE DESEMPENHO:")
    
    # Calcular médias de desempenho para cada configuração
    matriz_modelo = dados_preparados[model_name]
    matriz_limpa = matriz_modelo[~np.isnan(matriz_modelo).any(axis=1)]
    
    medias_configuracoes = {}
    for i in range(matriz_limpa.shape[1]):
        config_name = f"config_{i+1}"
        media_f1 = np.mean(matriz_limpa[:, i])
        medias_configuracoes[config_name] = media_f1
    
    # Sempre mostrar TOP 3 por desempenho
    ranking_desempenho = sorted(medias_configuracoes.items(), key=lambda x: x[1], reverse=True)
    
    print(f"   🏆 TOP 3 CONFIGURAÇÕES POR DESEMPENHO MÉDIO:")
    for i, (config, media) in enumerate(ranking_desempenho[:3], 1):
        medalha = "🥇" if i == 1 else "🥈" if i == 2 else "🥉"
        print(f"      {medalha} {config}: F1 = {media:.{precisao}f}")
    
    # Interpretação baseada nos testes estatísticos
    diferencas_significativas = (resultados['friedman_p_value'] is not None and 
                                resultados['friedman_p_value'] < 0.05 and 
                                resultados['pares_significativos'])
    
    if diferencas_significativas:
        melhor_config = ranking_desempenho[0]
        print(f"   🎯 RECOMENDAÇÃO: Use {melhor_config[0]} (F1 médio = {melhor_config[1]:.{precisao}f})")
        print("   ✅ As diferenças de desempenho são estatisticamente significativas")
        print("   💡 A primeira colocada é superior às demais")
    else:
        print("   🎯 RECOMENDAÇÃO: Use qualquer das top 3 configurações")
        if not diferencas_significativas:
            print("   💡 NÃO há diferenças significativas entre as configurações")
            print("   💡 Todas performam de forma estatisticamente similar")
        else:
            print("   ⚠️ Não foi possível determinar diferenças significativas devido a erro nos testes")
    
    print("\n" + "=" * 80)

print("\n🎉 ANÁLISE CONCLUÍDA!")
print("💡 Interpretação: p-valores < 0.05 indicam diferenças estatisticamente significativas")
print("📊 TOP 3 sempre mostrado por desempenho médio")


PROCESSANDO MODELO: Bagging
Dados válidos: 24 cenários × 12 configurações

Friedman Test:
  Estatística: 157.617471197316547205
  p-valor: 0.000000000000000000
  → DIFERENÇAS SIGNIFICATIVAS encontradas entre configurações
Aplicando teste de Nemenyi...
Resultado Nemenyi (p-valores ajustados):
                      config_1             config_2             config_3  \
config_1  1.000000000000000000 0.998699927757504158 0.999996365674858834   
config_2  0.998699927757504158 1.000000000000000000 0.999999616968251015   
config_3  0.999996365674858834 0.999999616968251015 1.000000000000000000   
config_4  0.000146392990452937 0.008536517386932174 0.001573535236024881   
config_5  0.000021459608138152 0.001865248396328400 0.000285714664579384   
config_6  0.000004737132320076 0.000545835359019975 0.000073447969950791   
config_7  0.004244349288529458 0.102002913481510804 0.028561814166093380   
config_8  0.004244349288529458 0.102002913481510804 0.028561814166093380   
config_9  0.0010203057

## Salvando os resultados

In [12]:
salvar_resultados_excel(resultados_testes, dados_preparados, f'resultados_testes_estatisticos_comites_{versao_execucao}.xlsx')

✅ Resultados salvos em: resultados_testes_estatisticos_comites_v20251102-1210.xlsx
📊 Arquivo contém:
   • Aba 'Resumo_Geral': 4 comites resumidos
   • Aba 'Pares_Significativos': 38 pares analisados
   • 4 abas de matrizes Nemenyi
